<a href="https://colab.research.google.com/github/Sanchita3005/Cyber-security-/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install flask

In [ ]:
from flask import Flask, render_template, request, redirect, session
import sqlite3
import hashlib
import logging
import re

app = Flask(__name__)
app.secret_key = "change-this-secret-key"

logging.basicConfig(
    filename="login.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def init_db():
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL
        )
    """)

    conn.commit()
    conn.close()

@app.route("/")
def home():
    return redirect("/login")

@app.route("/register", methods=["GET", "POST"])
def register():

    if request.method == "POST":
        username = request.form["username"].strip()
        password = request.form["password"]

        if not re.match(r"^[A-Za-z0-9_]{3,20}$", username):
            return "Invalid username!"

        if len(password) < 8:
            return "Password must contain at least 8 characters!"

        conn = sqlite3.connect("users.db")
        cursor = conn.cursor()

        try:
            cursor.execute(
                "INSERT INTO users (username, password) VALUES (?, ?)",
                (username, hash_password(password))
            )
            conn.commit()
        except sqlite3.IntegrityError:
            conn.close()
            return "Username already exists!"

        conn.close()

        logging.info("New user registered: %s", username)
        return redirect("/login")

    return render_template("register.html")


@app.route("/login", methods=["GET", "POST"])
def login():

    if request.method == "POST":

        username = request.form["username"].strip()
        password = request.form["password"]

        conn = sqlite3.connect("users.db")
        cursor = conn.cursor()

        cursor.execute(
            "SELECT * FROM users WHERE username = ? AND password = ?",
            (username, hash_password(password))
        )

        user = cursor.fetchone()
        conn.close()

        if user:
            session["username"] = username
            logging.info("Successful login: %s", username)
            return redirect("/dashboard")

        logging.warning("Failed login attempt: %s", username)
        return "Invalid username or password!"

    return render_template("login.html")


@app.route("/dashboard")
def dashboard():

    if "username" not in session:
        return redirect("/login")

    return render_template(
        "dashboard.html",
        username=session["username"]
    )


@app.route("/logout")
def logout():

    username = session.get("username")

    if username:
        logging.info("User logged out: %s", username)

    session.clear()

    return redirect("/login")


if __name__ == "__main__":
    init_db()
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


In [ ]:
<!DOCTYPE html>
<html>
<head>
    <title>Secure Login</title>
</head>
<body>

<h2>Secure Login System</h2>

<form method="POST">

    <label>Username:</label>
    <input type="text" name="username" required>

    <br><br>

    <label>Password:</label>
    <input type="password" name="password" required>

    <br><br>

    <button type="submit">Login</button>

</form>

<p>
    New user?
    <a href="/register">Register</a>
</p>

</body>
</html>

In [ ]:
<!DOCTYPE html>
<html>
<head>
    <title>Register</title>
</head>
<body>

<h2>Create Account</h2>

<form method="POST">

    <label>Username:</label>
    <input type="text" name="username" required>

    <br><br>

    <label>Password:</label>
    <input type="password" name="password" required>

    <br><br>

    <button type="submit">Register</button>

</form>

<p>
    Already registered?
    <a href="/login">Login</a>
</p>

</body>
</html>

In [ ]:
<!DOCTYPE html>
<html>
<head>
    <title>Dashboard</title>
</head>
<body>

<h2>Welcome, {{ username }}!</h2>

<p>You have successfully logged in.</p>

<a href="/logout">Logout</a>

</body>
</html>